In [ ]:
import duckdb
import os
from pathlib import Path

from openai import OpenAI
from dotenv import load_dotenv
import os
import plotly.express as px
import plotly.graph_objects as go
import json

load_dotenv()


In [ ]:
client = OpenAI(api_key=os.getenv('OPENAI_API_KEY'))


folder_path='../data/AdventureWorks'
db_path="../data/AdventureWorks.duckdb"

# conn = duckdb.connect(db_path)
conn = duckdb.connect(":memory:")

folder = Path(folder_path)

for csv_file in folder.glob("*.csv"):

    table_name = csv_file.stem  # file name without .csv

    query = f"""
    CREATE OR REPLACE TABLE {table_name} AS
    SELECT * FROM read_csv_auto('{csv_file}');
    """

    conn.execute(query)

    print(f"Created table: {table_name}")

conn.close()

In [ ]:
db_path="../data/AdventureWorks.duckdb"
# conn = duckdb.connect(db_path)
conn = duckdb.connect(":memory:")

# Inspect all tables
tables = conn.execute("""
    SELECT table_name
    FROM information_schema.tables
    WHERE table_schema = 'main'
""").fetchdf()

print(f"Found {len(tables)} tables:")
for t in tables["table_name"]:
    count = conn.execute(f"SELECT COUNT(*) FROM {t}").fetchone()[0]
    print(f"  {t} ({count} rows)")

In [ ]:
from pydantic import BaseModel, Field
from typing import Literal

# ── Model 1: Tool Input ──────────────────────────────────────────
# Validates what the agent sends TO the tool before it hits DuckDB
class RunSqlInput(BaseModel):
    sql: str = Field(..., description="Valid DuckDB SQL query")

# ── Model 2: Agent Final Answer ──────────────────────────────────
# Enforces the shape of what the agent returns TO you
class AgentAnswer(BaseModel):
    answer: str = Field(..., description="Plain English explanation of the result")
    sql_used: str = Field(..., description="The SQL that produced the final result")
    chart_type: Literal["bar", "line", "pie", "scatter", "histogram", "box", "none"]
    plotly_code: str = Field(..., description="Complete plotly figure code using df and fig variables")

print("✅ Models defined")
print(f"   RunSqlInput fields : {list(RunSqlInput.model_fields.keys())}")
print(f"   AgentAnswer fields : {list(AgentAnswer.model_fields.keys())}")


In [ ]:
TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "list_tables",
            "description": "List all available tables in the database. Always call this first.",
            "parameters": {
                "type": "object",
                "properties": {},
                "required": []
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_schema",
            "description": "Get schema and sample rows for a specific table. Call this for each relevant table before writing SQL.",
            "parameters": {
                "type": "object",
                "properties": {
                    "table_name": {
                        "type": "string",
                        "description": "Name of the table to inspect"
                    }
                },
                "required": ["table_name"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "run_sql",
            "description": "Run a DuckDB SQL query across one or more tables. Use JOINs when needed.",
            "parameters": {
                "type": "object",
                "properties": {
                    "sql": {
                        "type": "string",
                        "description": "Valid DuckDB SQL. No LIMIT on aggregations. LIMIT 100 on raw rows."
                    }
                },
                "required": ["sql"]
            }
        }
    }
]

In [ ]:
def execute_tool(tool_name: str, tool_args: dict) -> str:

    if tool_name == "list_tables":
        tables = conn.execute("""
            SELECT table_name
            FROM information_schema.tables
            WHERE table_schema = 'main'
        """).fetchdf()
        return tables["table_name"].tolist().__str__()

    elif tool_name == "get_schema":
        table = tool_args.get("table_name")

        # schema for specific table
        cols = conn.execute(f"DESCRIBE {table}").fetchdf()
        count = conn.execute(f"SELECT COUNT(*) FROM {table}").fetchone()[0]

        # sample 3 rows — helps agent understand join keys
        sample = conn.execute(f"SELECT * FROM {table} LIMIT 3").fetchdf()

        result = f"TABLE: {table} ({count} rows)\n"
        result += cols[["column_name", "column_type"]].to_string(index=False)
        result += f"\n\nSAMPLE:\n{sample.to_string(index=False)}"
        return result

    elif tool_name == "run_sql":
        try:
            validated = RunSqlInput(**tool_args)
            last_df = conn.execute(validated.sql).fetchdf()
            return last_df.to_string(index=False), last_df
        except Exception as e:
            return f"SQL_ERROR: {str(e)}", None

    return f"UNKNOWN_TOOL: {tool_name}"

In [ ]:
MAX_RETRIES = 3
AGENT_MODEL = "gpt-5-mini"
FINAL_MODEL = "gpt-5.4"

# SYSTEM_PROMPT = """You are a data analyst agent with access to a multi-table database.
#
# Follow this exact process:
# 1. Call list_tables to discover all available tables
# 2. Call get_schema for each relevant table
# 3. Identify join keys from column names and sample data
# 4. Write SQL with proper JOINs to answer the question
# 5. Repeat step 4 if needed
#
# SQL rules:
# - Use DuckDB SQL syntax
# - Always qualify column names with table name e.g. orders.customer_id
# - No LIMIT on GROUP BY queries
# - LIMIT 100 on raw row queries
# - Prefer LEFT JOIN unless INNER JOIN is clearly needed
#
# For plotly_code:
# - df is already loaded with your last query result
# - create a figure called fig
# - do NOT call fig.show()
# """

SYSTEM_PROMPT = """You are a data analyst agent with access to a multi-table database.

TOOL USAGE RULES:
- list_tables  → only call ONCE per conversation, skip if already in history
- get_schema   → only for tables directly needed, max 2-3 per question, skip if already fetched in this conversation
- run_sql      → required for ANY question asking for data, numbers, or graphs

QUESTION TYPES:
- "brief", "describe", "what tables" → list_tables only, no SQL
- "show", "graph", "how many", "top", "compare", "who", "which" → MUST run SQL

SQL rules:
- No LIMIT on GROUP BY queries
- LIMIT 100 on raw rows
- Always qualify column names with table name
- Prefer LEFT JOIN unless INNER JOIN clearly needed

For plotly_code:
- df is already loaded, create figure called fig, do NOT call fig.show()
"""


def run_agent(question: str) -> AgentAnswer:
    global conversation_history

    conversation_history.append({"role": "user", "content": question})
    messages = [{"role": "system", "content": SYSTEM_PROMPT}] + conversation_history

    print(f"\nQ: {question}")
    print("-" * 40)

    step = 0
    retries = 0
    last_df = None

    while True:
        response = client.chat.completions.create(
            model=AGENT_MODEL,
            messages=messages,
            tools=TOOLS
        )

        msg = response.choices[0].message
        messages.append(msg)
        conversation_history.append(msg)

        if not msg.tool_calls:
            break

        for tc in msg.tool_calls:
            step += 1
            args = json.loads(tc.function.arguments)

            if tc.function.name == "list_tables":
                print(f"Step {step}: list_tables()")
                result = execute_tool("list_tables", {})

            elif tc.function.name == "get_schema":
                print(f"Step {step}: get_schema({args.get('table_name')})")
                result = execute_tool("get_schema", args)

            elif tc.function.name == "run_sql":
                print(f"Step {step}: run_sql()")

                # ── Pydantic validation ───────────────────────────
                try:
                    validated = RunSqlInput(**args)
                except Exception as e:
                    result = f"VALIDATION_ERROR: {str(e)}"
                    print(f"  Validation failed: {e}")
                    retries += 1
                    if retries >= MAX_RETRIES:
                        print("  Max retries reached. Stopping.")
                        return None
                    tool_msg = {"role": "tool", "tool_call_id": tc.id, "content": result}
                    messages.append(tool_msg)
                    conversation_history.append(tool_msg)
                    continue

                print(f"  {validated.sql}")

                # ── SQL execution ─────────────────────────────────
                try:
                    last_df = conn.execute(validated.sql).fetchdf()
                    result = last_df.to_string(index=False)
                    retries = 0
                except Exception as e:
                    result = f"SQL_ERROR: {str(e)}"
                    print(f"  SQL failed: {e}")
                    retries += 1
                    if retries >= MAX_RETRIES:
                        print("  Max retries reached. Stopping.")
                        return None
                    print(f"  Retry {retries}/{MAX_RETRIES} — feeding error back to agent")

            tool_msg = {"role": "tool", "tool_call_id": tc.id, "content": result}
            messages.append(tool_msg)
            conversation_history.append(tool_msg)

    # ── Final structured answer ───────────────────────────────────
    try:
        df_columns = list(last_df.columns) if last_df is not None else []

        # only mention df if we actually have one
        df_context = (
            f"The final dataframe has these exact columns: {df_columns}. Use only these in plotly_code."
            if df_columns
            else "No SQL was run. Answer based on schema information only. Set chart_type to none."
        )

        final = client.beta.chat.completions.parse(
            model=FINAL_MODEL,
            messages=messages + [{
                "role": "user",
                "content": f"{df_context} Now give your final structured answer."
            }],
            response_format=AgentAnswer
        )
        result = final.choices[0].message.parsed

    except Exception as e:
        print(f"Final answer error: {e}")
        return None

    conversation_history.append({"role": "assistant", "content": result.answer})

    # ── Display ───────────────────────────────────────────────────
    print(f"\nAnswer: {result.answer}")
    print(f"Chart:  {result.chart_type}")

    if result.plotly_code and result.chart_type != "none" and last_df is not None:
        try:
            local_vars = {"df": last_df, "px": px, "go": go}
            exec(result.plotly_code, local_vars)
            local_vars["fig"].show()
        except Exception as e:
            print(f"Chart error: {e}")

    return result

In [ ]:
def reset_memory():
    global conversation_history
    conversation_history = []
    print("Memory cleared.")

reset_memory()

In [ ]:
run_agent("Hi, give me a brief about the data")

In [ ]:
run_agent("give me 3 bestselling products")

In [ ]:
run_agent("give me month by month  sales number vs revenue")

# memory

In [ ]:
import tiktoken

class MemoryManager:

    def __init__(self, max_turns: int = 6):
        self.max_turns = max_turns
        self.history = []       # user questions + agent answers only
        self.summary = None     # single rolling summary of older turns

    def add_question(self, question: str):
        self.history.append({"role": "user", "content": question})

    def add_answer(self, answer: str):
        self.history.append({"role": "assistant", "content": answer})
        self._maybe_summarise()

    def _maybe_summarise(self):
        """Triggered after every answer. Summarises when history exceeds max_turns."""
        if len(self.history) <= self.max_turns * 2:
            return

        # split
        keep_from = -(self.max_turns * 2)
        to_summarise = self.history[:keep_from]
        self.history = self.history[keep_from:]

        # include existing summary if present
        prior = f"Previous summary:\n{self.summary}\n\nNew turns:\n" if self.summary else ""
        text = prior + "\n".join(
            f"{m['role']}: {m['content']}"
            for m in to_summarise
            if isinstance(m.get("content"), str)
        )

        response = client.chat.completions.create(
            model=AGENT_MODEL,
            messages=[{
                "role": "user",
                "content": f"Summarise this conversation. Keep key questions, findings, numbers, insights. 4-5 sentences.\n\n{text}"
            }]
        )

        self.summary = response.choices[0].message.content
        print(f"  [memory] summarised — keeping last {self.max_turns} turns")

    def get_messages(self, system_prompt: str) -> list:
        system = [{"role": "system", "content": system_prompt}]

        summary_msg = []
        if self.summary:
            summary_msg = [{"role": "system", "content": f"[Earlier context]: {self.summary}"}]

        return system + summary_msg + self.history

    def reset(self):
        self.history = []
        self.summary = None
        print("  [memory] reset.")

    def stats(self):
        print(f"  [memory] turns: {len(self.history)//2}/{self.max_turns} | summary: {'yes' if self.summary else 'no'}")

In [ ]:
memory = MemoryManager(max_turns=6)


In [ ]:
MAX_RETRIES = 3
AGENT_MODEL = "gpt-5-mini"
FINAL_MODEL = "gpt-5.4"

# SYSTEM_PROMPT = """You are a data analyst agent with access to a multi-table database.
#
# Follow this exact process:
# 1. Call list_tables to discover all available tables
# 2. Call get_schema for each relevant table
# 3. Identify join keys from column names and sample data
# 4. Write SQL with proper JOINs to answer the question
# 5. Repeat step 4 if needed
#
# SQL rules:
# - Use DuckDB SQL syntax
# - Always qualify column names with table name e.g. orders.customer_id
# - No LIMIT on GROUP BY queries
# - LIMIT 100 on raw row queries
# - Prefer LEFT JOIN unless INNER JOIN is clearly needed
#
# For plotly_code:
# - df is already loaded with your last query result
# - create a figure called fig
# - do NOT call fig.show()
# """

SYSTEM_PROMPT = """You are a data analyst agent with access to a multi-table database.

TOOL USAGE RULES:
- list_tables  → only call ONCE per conversation, skip if already in history
- get_schema   → only for tables directly needed, max 2-3 per question, skip if already fetched in this conversation
- run_sql      → required for ANY question asking for data, numbers, or graphs

QUESTION TYPES:
- "brief", "describe", "what tables" → list_tables only, no SQL
- "show", "graph", "how many", "top", "compare", "who", "which" → MUST run SQL

SQL rules:
- No LIMIT on GROUP BY queries
- LIMIT 100 on raw rows
- Always qualify column names with table name
- Prefer LEFT JOIN unless INNER JOIN clearly needed

For plotly_code:
- df is already loaded, create figure called fig, do NOT call fig.show()
"""


def run_agent_with_memory(question: str) -> AgentAnswer:

    memory.add({"role": "user", "content": question})
    messages = memory.get_messages(SYSTEM_PROMPT)

    print(f"\nQ: {question}")
    print("-" * 40)

    step = 0
    retries = 0
    last_df = None

    while True:
        response = client.chat.completions.create(
            model=AGENT_MODEL,
            messages=messages,
            tools=TOOLS
        )

        msg = response.choices[0].message
        messages.append(msg)
        memory.add(msg)

        if not msg.tool_calls:
            break

        for tc in msg.tool_calls:
            step += 1
            args = json.loads(tc.function.arguments)

            if tc.function.name == "list_tables":
                print(f"Step {step}: list_tables()")
                result = execute_tool("list_tables", {})

            elif tc.function.name == "get_schema":
                print(f"Step {step}: get_schema({args.get('table_name')})")
                result = execute_tool("get_schema", args)

            elif tc.function.name == "run_sql":
                print(f"Step {step}: run_sql()")

                # ── Pydantic validation ───────────────────────────
                try:
                    validated = RunSqlInput(**args)
                except Exception as e:
                    result = f"VALIDATION_ERROR: {str(e)}"
                    print(f"  Validation failed: {e}")
                    retries += 1
                    if retries >= MAX_RETRIES:
                        print("  Max retries reached. Stopping.")
                        return None
                    tool_msg = {"role": "tool", "tool_call_id": tc.id, "content": result}
                    messages.append(tool_msg)
                    memory.add(tool_msg)
                    continue

                print(f"  {validated.sql}")

                # ── SQL execution ─────────────────────────────────
                try:
                    last_df = conn.execute(validated.sql).fetchdf()
                    result = last_df.to_string(index=False)
                    retries = 0
                except Exception as e:
                    result = f"SQL_ERROR: {str(e)}"
                    print(f"  SQL failed: {e}")
                    retries += 1
                    if retries >= MAX_RETRIES:
                        print("  Max retries reached. Stopping.")
                        return None
                    print(f"  Retry {retries}/{MAX_RETRIES} — feeding error back to agent")

            tool_msg = {"role": "tool", "tool_call_id": tc.id, "content": result}
            messages.append(tool_msg)
            memory.add(tool_msg)

    # ── Final structured answer ───────────────────────────────────
    try:
        df_columns = list(last_df.columns) if last_df is not None else []

        # only mention df if we actually have one
        df_context = (
            f"The final dataframe has these exact columns: {df_columns}. Use only these in plotly_code."
            if df_columns
            else "No SQL was run. Answer based on schema information only. Set chart_type to none."
        )

        final = client.beta.chat.completions.parse(
            model=FINAL_MODEL,
            messages=messages + [{
                "role": "user",
                "content": f"{df_context} Now give your final structured answer."
            }],
            response_format=AgentAnswer
        )
        result = final.choices[0].message.parsed

    except Exception as e:
        print(f"Final answer error: {e}")
        return None

    memory.add({"role": "assistant", "content": result.answer})

    # ── Display ───────────────────────────────────────────────────
    print(f"\nAnswer: {result.answer}")
    print(f"Chart:  {result.chart_type}")

    if result.plotly_code and result.chart_type != "none" and last_df is not None:
        try:
            local_vars = {"df": last_df, "px": px, "go": go}
            exec(result.plotly_code, local_vars)
            local_vars["fig"].show()
        except Exception as e:
            print(f"Chart error: {e}")

    return result

In [ ]:
run_agent_with_memory('show me monthly sales graph')

In [ ]:
memory.history

# Logging

In [ ]:
pwd

In [ ]:
import os
from loguru import logger

os.makedirs("../logs", exist_ok=True)

# ── Remove default handler, configure fresh ───────────────────────
logger.remove()

# ── Console — INFO and above ──────────────────────────────────────
logger.add(
    sink=lambda msg: print(msg, end=""),
    level="INFO",
    format="<green>{time:HH:mm:ss}</green> | <level>{level: <8}</level> | {message}"
)

# ── File — DEBUG and above (everything) ───────────────────────────
logger.add(
    sink="../logs/agent_{time:YYYY-MM-DD}.log",
    level="DEBUG",
    format="{time:YYYY-MM-DD HH:mm:ss} | {level: <8} | {message}",
    rotation="1 day",       # new file every day
    retention="7 days",     # keep last 7 days only
    compression="zip"       # compress old logs
)

In [ ]:
logger.debug("tool args received")        # file only
logger.info("agent started")              # console + file
logger.warning("retry attempt 1/3")       # console + file
logger.error("SQL failed")

In [ ]:
def run_agent_with_error_handling(question: str) -> AgentAnswer:

    memory.add({"role": "user", "content": question})
    messages = memory.get_messages(SYSTEM_PROMPT)

    logger.info(f"question: {question}")
    logger.debug(f"messages in context: {len(messages)}")
    memory.stats()

    step = 0
    retries = 0
    last_df = None

    while True:
        response = client.chat.completions.create(
            model=AGENT_MODEL,
            messages=messages,
            tools=TOOLS
        )

        msg = response.choices[0].message
        messages.append(msg)
        memory.add(msg)

        if not msg.tool_calls:
            break

        for tc in msg.tool_calls:
            step += 1
            args = json.loads(tc.function.arguments)

            if tc.function.name == "list_tables":
                logger.info(f"step {step}: list_tables()")
                result = execute_tool("list_tables", {})

            elif tc.function.name == "get_schema":
                logger.info(f"step {step}: get_schema({args.get('table_name')})")
                result = execute_tool("get_schema", args)

            elif tc.function.name == "run_sql":
                logger.info(f"step {step}: run_sql()")

                try:
                    validated = RunSqlInput(**args)
                except Exception as e:
                    logger.warning(f"validation failed: {e}")
                    retries += 1
                    result = f"VALIDATION_ERROR: {str(e)}"
                    if retries >= MAX_RETRIES:
                        logger.error("max retries reached. stopping.")
                        return None
                    tool_msg = {"role": "tool", "tool_call_id": tc.id, "content": result}
                    messages.append(tool_msg)
                    memory.add(tool_msg)
                    continue

                logger.info(f"sql: {validated.sql}")

                try:
                    last_df = conn.execute(validated.sql).fetchdf()
                    result = last_df.to_string(index=False)
                    logger.debug(f"rows returned: {len(last_df)}")
                    retries = 0
                except Exception as e:
                    logger.error(f"sql failed: {e}")
                    result = f"SQL_ERROR: {str(e)}"
                    retries += 1
                    if retries >= MAX_RETRIES:
                        logger.error("max retries reached. stopping.")
                        return None
                    logger.warning(f"retry {retries}/{MAX_RETRIES}")

            tool_msg = {"role": "tool", "tool_call_id": tc.id, "content": result}
            messages.append(tool_msg)
            memory.add(tool_msg)

    # ── Final structured answer ───────────────────────────────────
    try:
        df_columns = list(last_df.columns) if last_df is not None else []
        df_context = (
            f"DataFrame columns: {df_columns}. Use only these in plotly_code."
            if df_columns
            else "No SQL was run. Answer from schema only. Set chart_type to none."
        )

        final = client.beta.chat.completions.parse(
            model=FINAL_MODEL,
            messages=messages + [{
                "role": "user",
                "content": f"{df_context} Now give your final structured answer."
            }],
            response_format=AgentAnswer
        )
        result = final.choices[0].message.parsed

    except Exception as e:
        logger.error(f"final answer error: {e}")
        return None

    memory.add({"role": "assistant", "content": result.answer})

    logger.info(f"answer: {result.answer[:100]}...")
    logger.info(f"chart: {result.chart_type}")
    logger.debug(f"sql used: {result.sql_used}")

    if result.plotly_code and result.chart_type != "none" and last_df is not None:
        try:
            local_vars = {"df": last_df, "px": px, "go": go}
            exec(result.plotly_code, local_vars)
            local_vars["fig"].show()
        except Exception as e:
            logger.warning(f"chart error: {e}")

    return result

In [ ]:
run_agent_with_error_handling('show me the graphs of best seler product for each month')

## Logging: Langfuse

In [ ]:
# Cell 2 — load env vars once at the top of notebook
from dotenv import load_dotenv
import os

load_dotenv()

OPENAI_API_KEY      = os.getenv("OPENAI_API_KEY")
LANGFUSE_PUBLIC_KEY = os.getenv("LANGFUSE_PUBLIC_KEY")
LANGFUSE_SECRET_KEY = os.getenv("LANGFUSE_SECRET_KEY")
LANGFUSE_HOST       = os.getenv("LANGFUSE_HOST")

# verify all loaded
print("OPENAI_API_KEY:",      "✅" if OPENAI_API_KEY      else "❌ missing")
print("LANGFUSE_PUBLIC_KEY:", "✅" if LANGFUSE_PUBLIC_KEY else "❌ missing")
print("LANGFUSE_SECRET_KEY:", "✅" if LANGFUSE_SECRET_KEY else "❌ missing")

In [ ]:
from langfuse import Langfuse

langfuse = Langfuse(
    public_key=LANGFUSE_PUBLIC_KEY,
    secret_key=LANGFUSE_SECRET_KEY,
    host=LANGFUSE_HOST
)

# ── Test connection ───────────────────────────────────────────────
print("Langfuse connected:", langfuse.auth_check())

In [ ]:
import time

def run_agent_with_langfuse(question: str) -> AgentAnswer:

    # ── Start trace — one per question ───────────────────────────
    trace = langfuse.trace(
        name="agent-run",
        input=question,
        metadata={"model": AGENT_MODEL}
    )

    memory.add({"role": "user", "content": question})
    messages = memory.get_messages(SYSTEM_PROMPT)

    logger.info(f"question: {question}")
    logger.debug(f"messages in context: {len(messages)}")
    memory.stats()

    step = 0
    retries = 0
    last_df = None
    run_start = time.time()

    while True:
        response = client.chat.completions.create(
            model=AGENT_MODEL,
            messages=messages,
            tools=TOOLS
        )

        msg = response.choices[0].message
        messages.append(msg)
        memory.add(msg)

        if not msg.tool_calls:
            break

        for tc in msg.tool_calls:
            step += 1
            args = json.loads(tc.function.arguments)

            # ── Span — one per tool call ──────────────────────────
            span = trace.span(
                name=tc.function.name,
                input=args,
            )
            try:
                if tc.function.name == "list_tables":
                    logger.info(f"step {step}: list_tables()")
                    result = execute_tool("list_tables", {})

                elif tc.function.name == "get_schema":
                    logger.info(f"step {step}: get_schema({args.get('table_name')})")
                    result = execute_tool("get_schema", args)

                elif tc.function.name == "run_sql":
                    logger.info(f"step {step}: run_sql()")

                    try:
                        validated = RunSqlInput(**args)
                    except Exception as e:
                        logger.warning(f"validation failed: {e}")
                        result = f"VALIDATION_ERROR: {str(e)}"
                        retries += 1
                        span.end(output=result, level="WARNING")
                        if retries >= MAX_RETRIES:
                            logger.error("max retries reached. stopping.")
                            trace.update(output="max retries reached", level="ERROR")
                            return None
                        tool_msg = {"role": "tool", "tool_call_id": tc.id, "content": result}
                        messages.append(tool_msg)
                        memory.add(tool_msg)
                        continue

                    logger.info(f"  sql: {validated.sql}")

                    try:
                        last_df = conn.execute(validated.sql).fetchdf()
                        result = last_df.to_string(index=False)
                        logger.info(f"  rows returned: {len(last_df)}")
                        retries = 0
                    except Exception as e:
                        logger.error(f"sql failed: {e}")
                        result = f"SQL_ERROR: {str(e)}"
                        retries += 1
                        span.end(output=result, level="ERROR")
                        if retries >= MAX_RETRIES:
                            logger.error("max retries reached. stopping.")
                            trace.update(output="max retries reached", level="ERROR")
                            return None
                        logger.warning(f"retry {retries}/{MAX_RETRIES}")
            finally:
                # ── Close span ────────────────────────────────────────
                span.end(output=result[:500])   # cap output size

            tool_msg = {"role": "tool", "tool_call_id": tc.id, "content": result}
            messages.append(tool_msg)
            memory.add(tool_msg)

    # ── Final structured answer ───────────────────────────────────
    try:
        df_columns = list(last_df.columns) if last_df is not None else []
        df_context = (
            f"DataFrame columns: {df_columns}. Use only these in plotly_code."
            if df_columns
            else "No SQL was run. Answer from schema only. Set chart_type to none."
        )

        final = client.beta.chat.completions.parse(
            model=FINAL_MODEL,
            messages=messages + [{
                "role": "user",
                "content": f"{df_context} Now give your final structured answer."
            }],
            response_format=AgentAnswer
        )
        result = final.choices[0].message.parsed

    except Exception as e:
        logger.error(f"final answer error: {e}")
        trace.update(output=f"final answer error: {e}", level="ERROR")
        return None

    memory.add({"role": "assistant", "content": result.answer})

    duration = round(time.time() - run_start, 2)
    logger.info(f"answer: {result.answer[:100]}...")
    logger.info(f"chart: {result.chart_type}")
    logger.info(f"duration: {duration}s")

    # ── Close trace ───────────────────────────────────────────────
    trace.update(
        output=result.answer,
        metadata={
            "chart_type": result.chart_type,
            "duration_seconds": duration,
            "steps": step,
            "sql_used": result.sql_used
        }
    )
    langfuse.flush()    # ensure trace is sent before notebook moves on

    if result.plotly_code and result.chart_type != "none" and last_df is not None:
        try:
            local_vars = {"df": last_df, "px": px, "go": go}
            exec(result.plotly_code, local_vars)
            local_vars["fig"].show()
        except Exception as e:
            logger.warning(f"chart error: {e}")

    return result

In [ ]:
run_agent_with_langfuse('tell me about the routes of these products')

# Failure Handling & Rate Limits

In [ ]:
import time
import random
from openai import RateLimitError, APITimeoutError, APIConnectionError, APIStatusError

def call_openai(fn, *args, **kwargs):
    """
    Wraps any OpenAI API call with exponential backoff.
    Usage: call_openai(client.chat.completions.create, model=..., messages=...)
    """
    max_retries = 4
    base_wait   = 1     # seconds

    for attempt in range(max_retries):
        try:
            return fn(*args, **kwargs)

        except RateLimitError as e:
            wait = base_wait * (2 ** attempt) + random.uniform(0, 1)
            logger.warning(f"rate limit hit — waiting {wait:.1f}s (attempt {attempt+1}/{max_retries})")
            time.sleep(wait)

        except APITimeoutError as e:
            wait = base_wait * (2 ** attempt)
            logger.warning(f"timeout — waiting {wait:.1f}s (attempt {attempt+1}/{max_retries})")
            time.sleep(wait)

        except APIConnectionError as e:
            wait = base_wait * (2 ** attempt)
            logger.warning(f"connection error — waiting {wait:.1f}s (attempt {attempt+1}/{max_retries})")
            time.sleep(wait)

        except APIStatusError as e:
            # 500s are server errors — retry
            # 400s are client errors — don't retry
            if e.status_code >= 500:
                wait = base_wait * (2 ** attempt)
                logger.warning(f"server error {e.status_code} — waiting {wait:.1f}s")
                time.sleep(wait)
            else:
                logger.error(f"client error {e.status_code}: {e.message}")
                raise   # don't retry 400s — it won't help

        except Exception as e:
            logger.error(f"unexpected error: {type(e).__name__}: {e}")
            raise

    logger.error(f"all {max_retries} attempts failed")
    return None

In [ ]:
# ── Test ──────────────────────────────────────────────────────────
response = call_openai(
    client.chat.completions.create,
    model=AGENT_MODEL,
    messages=[{"role": "user", "content": "say hello"}],
    tools=TOOLS
)
print(response.choices[0].message.content)

In [ ]:
import time

def run_agent_with_failure_handling(question: str) -> AgentAnswer:

    # ── Start trace — one per question ───────────────────────────
    trace = langfuse.trace(
        name="agent-run",
        input=question,
        metadata={"model": AGENT_MODEL}
    )

    memory.add_question(question)
    messages = memory.get_messages(SYSTEM_PROMPT)

    logger.info(f"question: {question}")
    logger.debug(f"messages in context: {len(messages)}")
    memory.stats()

    step = 0
    retries = 0
    last_df = None
    run_start = time.time()

    while True:
        response = call_openai(
            client.chat.completions.create,
            model=AGENT_MODEL,
            messages=messages,
            tools=TOOLS
        )

        msg = response.choices[0].message
        messages.append(msg)
        # memory.add(msg)

        if not msg.tool_calls:
            break

        for tc in msg.tool_calls:
            step += 1
            args = json.loads(tc.function.arguments)

            # ── Span — one per tool call ──────────────────────────
            span = trace.span(
                name=tc.function.name,
                input=args,
            )
            try:
                if tc.function.name == "list_tables":
                    logger.info(f"step {step}: list_tables()")
                    result = execute_tool("list_tables", {})

                elif tc.function.name == "get_schema":
                    logger.info(f"step {step}: get_schema({args.get('table_name')})")
                    result = execute_tool("get_schema", args)

                elif tc.function.name == "run_sql":
                    logger.info(f"step {step}: run_sql()")

                    try:
                        validated = RunSqlInput(**args)
                    except Exception as e:
                        logger.warning(f"validation failed: {e}")
                        result = f"VALIDATION_ERROR: {str(e)}"
                        retries += 1
                        span.end(output=result, level="WARNING")
                        if retries >= MAX_RETRIES:
                            logger.error("max retries reached. stopping.")
                            trace.update(output="max retries reached", level="ERROR")
                            return None
                        tool_msg = {"role": "tool", "tool_call_id": tc.id, "content": result}
                        messages.append(tool_msg)
                        # memory.add(tool_msg)
                        continue

                    logger.info(f"  sql: {validated.sql}")

                    try:
                        last_df = conn.execute(validated.sql).fetchdf()
                        result = last_df.to_string(index=False)
                        logger.info(f"  rows returned: {len(last_df)}")
                        retries = 0
                    except Exception as e:
                        logger.error(f"sql failed: {e}")
                        result = f"SQL_ERROR: {str(e)}"
                        retries += 1
                        span.end(output=result, level="ERROR")
                        if retries >= MAX_RETRIES:
                            logger.error("max retries reached. stopping.")
                            trace.update(output="max retries reached", level="ERROR")
                            return None
                        logger.warning(f"retry {retries}/{MAX_RETRIES}")
            finally:
                # ── Close span ────────────────────────────────────────
                span.end(output=result[:500])   # cap output size

            tool_msg = {"role": "tool", "tool_call_id": tc.id, "content": result}
            messages.append(tool_msg)
            # memory.add(tool_msg)


    # Recheck tokens
    messages = memory.get_messages(SYSTEM_PROMPT)

    # ── Final structured answer ───────────────────────────────────
    try:
        df_columns = list(last_df.columns) if last_df is not None else []
        df_context = (
            f"DataFrame columns: {df_columns}. Use only these in plotly_code."
            if df_columns
            else "No SQL was run. Answer from schema only. Set chart_type to none."
        )

        final = call_openai(
            client.beta.chat.completions.parse,
            model=FINAL_MODEL,
            messages=messages + [{
                "role": "user",
                "content": f"{df_context} Now give your final structured answer."
            }],
            response_format=AgentAnswer
        )

        result = final.choices[0].message.parsed

    except Exception as e:
        logger.error(f"final answer error: {e}")
        trace.update(output=f"final answer error: {e}", level="ERROR")
        return None

    # memory.add({"role": "assistant", "content": result.answer})
    memory.add_answer(result.answer)

    duration = round(time.time() - run_start, 2)
    logger.info(f"answer: {result.answer[:100]}...")
    logger.info(f"chart: {result.chart_type}")
    logger.info(f"duration: {duration}s")

    # ── Close trace ───────────────────────────────────────────────
    trace.update(
        output=result.answer,
        metadata={
            "chart_type": result.chart_type,
            "duration_seconds": duration,
            "steps": step,
            "sql_used": result.sql_used
        }
    )
    langfuse.flush()    # ensure trace is sent before notebook moves on

    if result.plotly_code and result.chart_type != "none" and last_df is not None:
        try:
            local_vars = {"df": last_df, "px": px, "go": go}
            exec(result.plotly_code, local_vars)
            local_vars["fig"].show()
        except Exception as e:
            logger.warning(f"chart error: {e}")

    return result

In [ ]:
run_agent_with_failure_handling('hlp me with workorder of this product')


## memory correction

In [ ]:
import tiktoken

class MemoryManager:

    def __init__(self, model: str, max_turns: int = 6, max_tokens: int = 6000):
        self.model      = model
        self.max_turns  = max_turns
        self.max_tokens = max_tokens
        self.encoder    = tiktoken.encoding_for_model(model)
        self.history     = []    # Q&A pairs only
        self.summary     = None  # rolling summary of older turns
        self.tables_cache = None # list_tables result
        self.schema_cache = {}   # {table_name: schema string}

    # ── Token counting ────────────────────────────────────────────
    def count_tokens(self, messages: list) -> int:
        return sum(
            len(self.encoder.encode(m["content"]))
            for m in messages
            if isinstance(m.get("content"), str)
        )

    # ── Cache ─────────────────────────────────────────────────────
    def cache_tables(self, result: str):
        self.tables_cache = result
        logger.info(f"  [memory] tables cached")

    def cache_schema(self, table_name: str, result: str):
        self.schema_cache[table_name] = result
        logger.info(f"  [memory] schema cached: {table_name}")

    def get_cached_tables(self) -> str | None:
        return self.tables_cache

    def get_cached_schema(self, table_name: str) -> str | None:
        return self.schema_cache.get(table_name)

    # ── Q&A ───────────────────────────────────────────────────────
    def add_question(self, question: str):
        self.history.append({"role": "user", "content": question})

    def add_answer(self, answer: str):
        self.history.append({"role": "assistant", "content": answer})
        self._maybe_summarise()

    def _maybe_summarise(self):
        """Triggered after every answer. Acts only when turns exceed max."""
        turns = len([m for m in self.history if m["role"] == "user"])
        if turns <= self.max_turns:
            return

        # split — keep last max_turns, summarise the rest
        keep_from = -(self.max_turns * 2)
        to_summarise = self.history[:keep_from]
        self.history  = self.history[keep_from:]

        prior = f"Previous summary:\n{self.summary}\n\nNew turns:\n" if self.summary else ""
        text  = prior + "\n".join(
            f"{m['role']}: {m['content']}"
            for m in to_summarise
            if isinstance(m.get("content"), str)
        )

        tokens_before = self.count_tokens(self.get_messages(""))
        logger.info(f"  [memory] summarising — tokens before: {tokens_before}")

        response = client.chat.completions.create(
            model=self.model,
            messages=[{
                "role": "user",
                "content": f"Summarise this conversation. Keep key questions, findings, numbers, insights. 4-5 sentences.\n\n{text}"
            }]
        )
        self.summary = response.choices[0].message.content

        tokens_after = self.count_tokens(self.get_messages(""))
        logger.info(f"  [memory] summarised — tokens after: {tokens_after}")

    # ── Build messages ────────────────────────────────────────────
    def get_messages(self, system_prompt: str) -> list:
        messages = [{"role": "system", "content": system_prompt}]

        if self.tables_cache:
            messages.append({
                "role": "system",
                "content": f"[Available tables]: {self.tables_cache}"
            })

        if self.schema_cache:
            schema_text = "\n\n".join(
                f"[{t}]:\n{s}"
                for t, s in self.schema_cache.items()
            )
            messages.append({
                "role": "system",
                "content": f"[Table schemas]:\n{schema_text}"
            })

        if self.summary:
            messages.append({
                "role": "system",
                "content": f"[Earlier context]: {self.summary}"
            })

        messages.extend(self.history)
        return messages

    # ── Stats ─────────────────────────────────────────────────────
    def stats(self, system_prompt: str = ""):
        tokens = self.count_tokens(self.get_messages(system_prompt))
        turns  = len([m for m in self.history if m["role"] == "user"])
        print(f"  [memory] turns: {turns} | cached schemas: {len(self.schema_cache)} | summary: {'yes' if self.summary else 'no'} | tokens: {tokens}/{self.max_tokens}")

    def reset(self):
        self.history  = []
        self.summary  = None
        # cache preserved intentionally
        print("  [memory] reset — cache preserved")

In [ ]:
memory = MemoryManager(model=AGENT_MODEL, max_turns=2, max_tokens=6000)


In [ ]:
SYSTEM_PROMPT = """You are a data analyst agent with access to a DuckDB database.

CONTEXT RULES:
- If [Available tables] is in context → do NOT call list_tables
- If [Table schemas] contains the table you need → do NOT call get_schema
- Always run SQL to answer data questions — never guess from context alone

PROCESS:
1. Check context for available schema
2. Fetch only missing schemas
3. Always run_sql for any data or comparison question
4. Return final structured answer

SQL rules:
- No LIMIT on GROUP BY queries
- LIMIT 100 on raw rows
- Always qualify column names with table name
- Prefer LEFT JOIN unless INNER JOIN clearly needed
- Never assume dates or years — always query what exists first

plotly_code rules:
- df is already loaded as a pandas DataFrame
- create figure called fig
- do NOT call fig.show()
- use double quotes only — never single quotes
"""

In [ ]:
def execute_tool(tool_name: str, tool_args: dict) -> tuple:
    """Returns (result_string, dataframe_or_none)"""

    if tool_name == "list_tables":
        cached = memory.get_cached_tables()
        if cached:
            logger.info("  list_tables: from cache")
            return cached, None
        tables = conn.execute("""
            SELECT table_name
            FROM information_schema.tables
            WHERE table_schema = 'main'
        """).fetchdf()["table_name"].tolist()
        result = str(tables)
        memory.cache_tables(result)
        return result, None

    elif tool_name == "get_schema":
        table = tool_args.get("table_name")
        cached = memory.get_cached_schema(table)
        if cached:
            logger.info(f"  get_schema({table}): from cache")
            return cached, None
        cols   = conn.execute(f"DESCRIBE {table}").fetchdf()
        count  = conn.execute(f"SELECT COUNT(*) FROM {table}").fetchone()[0]
        sample = conn.execute(f"SELECT * FROM {table} LIMIT 1").fetchdf()
        result = (
            f"TABLE: {table} | rows: {count}\n"
            f"COLUMNS:\n{cols[['column_name','column_type']].to_string(index=False)}\n"
            f"SAMPLE:\n{sample.to_string(index=False)}"
        )
        memory.cache_schema(table, result)
        return result, None

    elif tool_name == "run_sql":
        try:
            validated = RunSqlInput(**tool_args)
            df = conn.execute(validated.sql).fetchdf()
            return df.to_string(index=False), df
        except Exception as e:
            return f"SQL_ERROR: {str(e)}", None

    return f"UNKNOWN_TOOL: {tool_name}", None

In [ ]:
import time

MAX_RETRIES  = 3
AGENT_MODEL  = "gpt-4.1-mini"
FINAL_MODEL  = "gpt-5-mini"

memory = MemoryManager(model=AGENT_MODEL, max_turns=2, max_tokens=6000)


def run_agent(question: str) -> AgentAnswer:

    # ── Trace ─────────────────────────────────────────────────────
    trace = langfuse.trace(
        name="agent-run",
        input=question,
        metadata={"model": AGENT_MODEL}
    )

    # ── Memory ────────────────────────────────────────────────────
    memory.add_question(question)
    messages = memory.get_messages(SYSTEM_PROMPT)

    logger.info(f"question: {question}")
    memory.stats(SYSTEM_PROMPT)

    step     = 0
    retries  = 0
    last_df  = None
    run_start = time.time()

    while True:
        response = call_openai(
            client.chat.completions.create,
            model=AGENT_MODEL,
            messages=messages,
            tools=TOOLS
        )
        if response is None:
            logger.error("agent loop: all retries failed")
            trace.update(output="openai call failed")
            return None

        msg = response.choices[0].message
        messages.append(msg)

        if not msg.tool_calls:
            break

        for tc in msg.tool_calls:
            step += 1
            args = json.loads(tc.function.arguments)
            span = trace.span(name=tc.function.name, input=args)

            try:
                if tc.function.name == "list_tables":
                    logger.info(f"step {step}: list_tables()")
                    result, _ = execute_tool("list_tables", {})

                elif tc.function.name == "get_schema":
                    logger.info(f"step {step}: get_schema({args.get('table_name')})")
                    result, _ = execute_tool("get_schema", args)

                elif tc.function.name == "run_sql":
                    logger.info(f"step {step}: run_sql()")

                    try:
                        validated = RunSqlInput(**args)
                    except Exception as e:
                        logger.warning(f"  validation failed: {e}")
                        result   = f"VALIDATION_ERROR: {str(e)}"
                        retries += 1
                        if retries >= MAX_RETRIES:
                            logger.error("max retries reached. stopping.")
                            trace.update(output="max retries reached")
                            return None
                        messages.append({"role": "tool", "tool_call_id": tc.id, "content": result})
                        continue

                    logger.info(f"  sql: {validated.sql}")
                    result, last_df = execute_tool("run_sql", {"sql": validated.sql})

                    if last_df is not None:
                        logger.info(f"  rows returned: {len(last_df)}")
                        retries = 0
                    else:
                        logger.error(f"  sql failed: {result}")
                        retries += 1
                        if retries >= MAX_RETRIES:
                            logger.error("max retries reached. stopping.")
                            trace.update(output="max retries reached")
                            return None
                        logger.warning(f"  retry {retries}/{MAX_RETRIES}")

            finally:
                span.end(output=str(result)[:500])

            # tool result stays in messages this turn only — never in memory
            messages.append({"role": "tool", "tool_call_id": tc.id, "content": result})

    # ── Final answer ──────────────────────────────────────────────
    try:
        df_columns = [str(col) for col in last_df.columns] if last_df is not None else []
        df_context = (
            f"DataFrame columns: {df_columns}. Use only these in plotly_code."
            if df_columns
            else "No SQL was run. Answer from context only. Set chart_type to none."
        )

        final = call_openai(
            client.beta.chat.completions.parse,
            model=FINAL_MODEL,
            messages=messages + [{
                "role": "user",
                "content": f"{df_context} Now give your final structured answer."
            }],
            response_format=AgentAnswer
        )
        if final is None:
            logger.error("final answer: all retries failed")
            trace.update(output="final answer call failed")
            return None

        result = final.choices[0].message.parsed

    except Exception as e:
        logger.error(f"final answer error: {e}")
        # fallback — no chart
        try:
            fallback = call_openai(
                client.beta.chat.completions.parse,
                model=FINAL_MODEL,
                messages=messages + [{
                    "role": "user",
                    "content": f"{df_context} Give final answer. Set plotly_code to empty string and chart_type to none."
                }],
                response_format=AgentAnswer
            )
            if fallback is None:
                return None
            result = fallback.choices[0].message.parsed
            logger.warning("fallback answer used — no chart")
        except Exception as e2:
            logger.error(f"fallback failed: {e2}")
            trace.update(output=f"final answer error: {e2}")
            return None

    # ── Memory — Q&A only ─────────────────────────────────────────
    memory.add_answer(result.answer)

    duration = round(time.time() - run_start, 2)
    logger.info(f"answer: {result.answer[:100]}...")
    logger.info(f"chart:  {result.chart_type}")
    logger.info(f"duration: {duration}s")

    trace.update(
        output=result.answer,
        metadata={
            "chart_type":       result.chart_type,
            "duration_seconds": duration,
            "steps":            step,
            "sql_used":         result.sql_used
        }
    )
    langfuse.flush()

    if result.plotly_code and result.chart_type != "none" and last_df is not None:
        try:
            local_vars = {"df": last_df, "px": px, "go": go}
            exec(result.plotly_code, local_vars)
            local_vars["fig"].show()
        except Exception as e:
            logger.warning(f"chart error: {e}")

    return result

In [ ]:
run_agent("which product has the most transactions?")

In [ ]:
run_agent("this products month on month transactions")

In [ ]:
run_agent("lets compare this product sales vs others in nov month")

## evaluation

In [ ]:
from pydantic import BaseModel, Field

class EvaluationResult(BaseModel):
    answer_relevance:       float = Field(..., ge=0, le=1, description="Did the answer address the question?")
    sql_correctness:        float = Field(..., ge=0, le=1, description="Did the SQL match the question intent?")
    chart_appropriateness:  float = Field(..., ge=0, le=1, description="Was the chart type appropriate for the data?")
    tool_efficiency:        float = Field(..., ge=0, le=1, description="Were minimum tool calls used?")
    reasoning:              str   = Field(..., description="Brief explanation of scores")


def evaluate_run(
    question:   str,
    answer:     str,
    sql_used:   str,
    chart_type: str,
    steps:      int
) -> EvaluationResult:
    """
    LLM-as-a-judge. Uses cheap model — evaluation does not need gpt-4o.
    Returns structured scores for each dimension.
    """

    prompt = f"""You are evaluating a data analyst AI agent. Score each dimension from 0 to 1.

QUESTION ASKED:
{question}

AGENT ANSWER:
{answer}

SQL USED:
{sql_used}

CHART TYPE CHOSEN:
{chart_type}

TOOL CALLS MADE:
{steps} steps total

SCORING GUIDE:
answer_relevance:
  1.0 = directly and completely answers the question
  0.5 = partially answers, missing key details
  0.0 = irrelevant or wrong answer

sql_correctness:
  1.0 = SQL clearly matches the question intent
  0.5 = SQL runs but may miss edge cases
  0.0 = SQL is wrong or missing entirely

chart_appropriateness:
  1.0 = perfect chart type for this data
  0.5 = acceptable but not ideal
  0.0 = wrong chart type or none when needed

tool_efficiency:
  1.0 = minimum tools used, no redundant calls
  0.5 = some redundant calls but acceptable
  0.0 = excessive redundant tool calls

Score honestly. Be strict."""

    response = call_openai(
        client.beta.chat.completions.parse,
        model=AGENT_MODEL,    # cheap model for evaluation
        messages=[{"role": "user", "content": prompt}],
        response_format=EvaluationResult
    )

    return response.choices[0].message.parsed

In [ ]:
# ── Test ──────────────────────────────────────────────────────────
test_eval = evaluate_run(
    question   = "which product has the most transactions?",
    answer     = "Water Bottle - 30 oz. has the most transactions with 3688.",
    sql_used   = "SELECT p.Name, COUNT(*) FROM TransactionHistory t JOIN Product p ON t.ProductID = p.ProductID GROUP BY p.Name ORDER BY COUNT(*) DESC LIMIT 1",
    chart_type = "bar",
    steps      = 3
)

print(f"answer_relevance:      {test_eval.answer_relevance}")
print(f"sql_correctness:       {test_eval.sql_correctness}")
print(f"chart_appropriateness: {test_eval.chart_appropriateness}")
print(f"tool_efficiency:       {test_eval.tool_efficiency}")
print(f"reasoning:             {test_eval.reasoning}")

In [ ]:
import time

MAX_RETRIES  = 3
AGENT_MODEL  = "gpt-4.1-mini"
FINAL_MODEL  = "gpt-5-mini"

memory = MemoryManager(model=AGENT_MODEL, max_turns=2, max_tokens=6000)


def run_agent_with_evaluation(question: str) -> AgentAnswer:

    # ── Trace ─────────────────────────────────────────────────────
    trace = langfuse.trace(
        name="agent-run",
        input=question,
        metadata={"model": AGENT_MODEL}
    )

    # ── Memory ────────────────────────────────────────────────────
    memory.add_question(question)
    messages = memory.get_messages(SYSTEM_PROMPT)

    logger.info(f"question: {question}")
    memory.stats(SYSTEM_PROMPT)

    step     = 0
    retries  = 0
    last_df  = None
    run_start = time.time()

    while True:
        response = call_openai(
            client.chat.completions.create,
            model=AGENT_MODEL,
            messages=messages,
            tools=TOOLS
        )
        if response is None:
            logger.error("agent loop: all retries failed")
            trace.update(output="openai call failed")
            return None

        msg = response.choices[0].message
        messages.append(msg)

        if not msg.tool_calls:
            break

        for tc in msg.tool_calls:
            step += 1
            args = json.loads(tc.function.arguments)
            span = trace.span(name=tc.function.name, input=args)

            try:
                if tc.function.name == "list_tables":
                    logger.info(f"step {step}: list_tables()")
                    result, _ = execute_tool("list_tables", {})

                elif tc.function.name == "get_schema":
                    logger.info(f"step {step}: get_schema({args.get('table_name')})")
                    result, _ = execute_tool("get_schema", args)

                elif tc.function.name == "run_sql":
                    logger.info(f"step {step}: run_sql()")

                    try:
                        validated = RunSqlInput(**args)
                    except Exception as e:
                        logger.warning(f"  validation failed: {e}")
                        result   = f"VALIDATION_ERROR: {str(e)}"
                        retries += 1
                        if retries >= MAX_RETRIES:
                            logger.error("max retries reached. stopping.")
                            trace.update(output="max retries reached")
                            return None
                        messages.append({"role": "tool", "tool_call_id": tc.id, "content": result})
                        continue

                    logger.info(f"  sql: {validated.sql}")
                    result, last_df = execute_tool("run_sql", {"sql": validated.sql})

                    if last_df is not None:
                        logger.info(f"  rows returned: {len(last_df)}")
                        retries = 0
                    else:
                        logger.error(f"  sql failed: {result}")
                        retries += 1
                        if retries >= MAX_RETRIES:
                            logger.error("max retries reached. stopping.")
                            trace.update(output="max retries reached")
                            return None
                        logger.warning(f"  retry {retries}/{MAX_RETRIES}")

            finally:
                span.end(output=str(result)[:500])

            # tool result stays in messages this turn only — never in memory
            messages.append({"role": "tool", "tool_call_id": tc.id, "content": result})

    # ── Final answer ──────────────────────────────────────────────
    try:
        df_columns = [str(col) for col in last_df.columns] if last_df is not None else []
        df_context = (
            f"DataFrame columns: {df_columns}. Use only these in plotly_code."
            if df_columns
            else "No SQL was run. Answer from context only. Set chart_type to none."
        )

        final = call_openai(
            client.beta.chat.completions.parse,
            model=FINAL_MODEL,
            messages=messages + [{
                "role": "user",
                "content": f"{df_context} Now give your final structured answer."
            }],
            response_format=AgentAnswer
        )
        if final is None:
            logger.error("final answer: all retries failed")
            trace.update(output="final answer call failed")
            return None

        result = final.choices[0].message.parsed

    except Exception as e:
        logger.error(f"final answer error: {e}")
        # fallback — no chart
        try:
            fallback = call_openai(
                client.beta.chat.completions.parse,
                model=FINAL_MODEL,
                messages=messages + [{
                    "role": "user",
                    "content": f"{df_context} Give final answer. Set plotly_code to empty string and chart_type to none."
                }],
                response_format=AgentAnswer
            )
            if fallback is None:
                return None
            result = fallback.choices[0].message.parsed
            logger.warning("fallback answer used — no chart")
        except Exception as e2:
            logger.error(f"fallback failed: {e2}")
            trace.update(output=f"final answer error: {e2}")
            return None

    # ── Memory — Q&A only ─────────────────────────────────────────
    memory.add_answer(result.answer)

    duration = round(time.time() - run_start, 2)
    logger.info(f"answer: {result.answer[:100]}...")
    logger.info(f"chart:  {result.chart_type}")
    logger.info(f"duration: {duration}s")

    trace.update(
        output=result.answer,
        metadata={
            "chart_type":       result.chart_type,
            "duration_seconds": duration,
            "steps":            step,
            "sql_used":         result.sql_used
        }
    )
    langfuse.flush()

    # ── Evaluation ────────────────────────────────────────────────
    try:
        eval_result = evaluate_run(
            question   = question,
            answer     = result.answer,
            sql_used   = result.sql_used,
            chart_type = result.chart_type,
            steps      = step
        )

        # send scores to langfuse trace
        trace.score(name="answer_relevance",      value=eval_result.answer_relevance)
        trace.score(name="sql_correctness",       value=eval_result.sql_correctness)
        trace.score(name="chart_appropriateness", value=eval_result.chart_appropriateness)
        trace.score(name="tool_efficiency",       value=eval_result.tool_efficiency)

        langfuse.flush()

        logger.info(f"eval — relevance: {eval_result.answer_relevance} | sql: {eval_result.sql_correctness} | chart: {eval_result.chart_appropriateness} | efficiency: {eval_result.tool_efficiency}")
        logger.debug(f"eval reasoning: {eval_result.reasoning}")

    except Exception as e:
        logger.warning(f"evaluation failed: {e}")

    if result.plotly_code and result.chart_type != "none" and last_df is not None:
        try:
            local_vars = {"df": last_df, "px": px, "go": go}
            exec(result.plotly_code, local_vars)
            local_vars["fig"].show()
        except Exception as e:
            logger.warning(f"chart error: {e}")

    return result

In [ ]:
run_agent_with_evaluation('top 3 transacted prodcuts in dec')

## cost optimisation

In [ ]:
class CostTracker:

    def __init__(self):
        self.input_tokens  = 0
        self.output_tokens = 0

    def add(self, input_tokens: int, output_tokens: int):
        self.input_tokens  += input_tokens
        self.output_tokens += output_tokens

    def log(self):
        logger.info(f"cost — input: {self.input_tokens} | output: {self.output_tokens} tokens")

    def reset(self):
        self.input_tokens  = 0
        self.output_tokens = 0

In [ ]:
# ── Constants — top of notebook ───────────────────────────────────
MODEL_COSTS = {
    "gpt-4.1-mini": {"input": 0.4,  "output": 1.60},
    "gpt-5-mini":  {"input": 0.25,  "output": 2.00},
    "gpt-5.4":     {"input": 2.50,  "output": 15.00},
}

# ── Initialise once ───────────────────────────────────────────────
cost_tracker = CostTracker()

In [ ]:
# ── Test ──────────────────────────────────────────────────────────
cost_tracker.track("gpt-4o-mini", input_tokens=500, output_tokens=100)
cost_tracker.track("gpt-4o",      input_tokens=200, output_tokens=50)
cost_tracker.log()
print(cost_tracker.summary())
cost_tracker.reset()

In [ ]:
def call_openai(fn, *args, **kwargs):
    max_retries = 4
    base_wait   = 1

    for attempt in range(max_retries):
        try:
            response = fn(*args, **kwargs)
            return response

        except RateLimitError as e:
            wait = base_wait * (2 ** attempt) + random.uniform(0, 1)
            logger.warning(f"rate limit — waiting {wait:.1f}s (attempt {attempt+1}/{max_retries})")
            time.sleep(wait)

        except APITimeoutError:
            wait = base_wait * (2 ** attempt)
            logger.warning(f"timeout — waiting {wait:.1f}s (attempt {attempt+1}/{max_retries})")
            time.sleep(wait)

        except APIConnectionError:
            wait = base_wait * (2 ** attempt)
            logger.warning(f"connection error — waiting {wait:.1f}s (attempt {attempt+1}/{max_retries})")
            time.sleep(wait)

        except APIStatusError as e:
            if e.status_code >= 500:
                wait = base_wait * (2 ** attempt)
                logger.warning(f"server error {e.status_code} — waiting {wait:.1f}s")
                time.sleep(wait)
            else:
                logger.error(f"client error {e.status_code}: {e.message}")
                raise

        except Exception as e:
            logger.error(f"unexpected error: {type(e).__name__}: {e}")
            raise

    logger.error(f"all {max_retries} attempts failed")
    return None

In [ ]:
import time

MAX_RETRIES = 3
AGENT_MODEL = "gpt-4.1-mini"
FINAL_MODEL = "gpt-5-mini"

memory = MemoryManager(model=AGENT_MODEL, max_turns=2, max_tokens=6000)


def run_agent_with_cost(question: str) -> AgentAnswer:

    # ── Trace ─────────────────────────────────────────────────────
    trace = langfuse.trace(
        name="agent-run",
        input=question,
        metadata={"model": AGENT_MODEL}
    )

    # ── Memory ────────────────────────────────────────────────────
    memory.add_question(question)
    messages = memory.get_messages(SYSTEM_PROMPT)

    logger.info(f"question: {question}")
    memory.stats(SYSTEM_PROMPT)

    step      = 0
    retries   = 0
    last_df   = None
    run_start = time.time()

    while True:

        # ── LLM call — generation for cost tracking ───────────────
        gen = trace.generation(
            name  = f"agent-loop-{step}",
            model = AGENT_MODEL,
            input = messages
        )

        response = call_openai(
            client.chat.completions.create,
            model=AGENT_MODEL,
            messages=messages,
            tools=TOOLS
        )

        if response is None:
            logger.error("agent loop: all retries failed")
            gen.end(output="failed")
            trace.update(output="openai call failed")
            return None

        msg = response.choices[0].message
        messages.append(msg)

        gen.end(
            output=msg.content or "",
            usage={
                "input":  response.usage.prompt_tokens,
                "output": response.usage.completion_tokens
            }
        )

        if not msg.tool_calls:
            break

        for tc in msg.tool_calls:
            step += 1
            args = json.loads(tc.function.arguments)

            # ── Tool call — span ──────────────────────────────────
            span = trace.span(name=tc.function.name, input=args)

            try:
                if tc.function.name == "list_tables":
                    logger.info(f"step {step}: list_tables()")
                    result, _ = execute_tool("list_tables", {})

                elif tc.function.name == "get_schema":
                    logger.info(f"step {step}: get_schema({args.get('table_name')})")
                    result, _ = execute_tool("get_schema", args)

                elif tc.function.name == "run_sql":
                    logger.info(f"step {step}: run_sql()")

                    try:
                        validated = RunSqlInput(**args)
                    except Exception as e:
                        logger.warning(f"  validation failed: {e}")
                        result   = f"VALIDATION_ERROR: {str(e)}"
                        retries += 1
                        if retries >= MAX_RETRIES:
                            logger.error("max retries reached. stopping.")
                            trace.update(output="max retries reached")
                            return None
                        messages.append({"role": "tool", "tool_call_id": tc.id, "content": result})
                        continue

                    logger.info(f"  sql: {validated.sql}")
                    result, last_df = execute_tool("run_sql", {"sql": validated.sql})

                    if last_df is not None:
                        logger.info(f"  rows returned: {len(last_df)}")
                        retries = 0
                    else:
                        logger.error(f"  sql failed: {result}")
                        retries += 1
                        if retries >= MAX_RETRIES:
                            logger.error("max retries reached. stopping.")
                            trace.update(output="max retries reached")
                            return None
                        logger.warning(f"  retry {retries}/{MAX_RETRIES}")

            finally:
                span.end(output=str(result)[:500])

            messages.append({"role": "tool", "tool_call_id": tc.id, "content": result})

    # ── Final answer ──────────────────────────────────────────────
    try:
        df_columns = [str(col) for col in last_df.columns] if last_df is not None else []
        df_context = (
            f"DataFrame columns: {df_columns}. Use only these in plotly_code."
            if df_columns
            else "No SQL was run. Answer from context only. Set chart_type to none."
        )

        gen_final = trace.generation(
            name  = "final-answer",
            model = FINAL_MODEL,
            input = messages
        )

        final = call_openai(
            client.beta.chat.completions.parse,
            model=FINAL_MODEL,
            messages=messages + [{
                "role": "user",
                "content": f"{df_context} Now give your final structured answer."
            }],
            response_format=AgentAnswer
        )

        if final is None:
            logger.error("final answer: all retries failed")
            gen_final.end(output="failed")
            trace.update(output="final answer call failed")
            return None

        result = final.choices[0].message.parsed

        gen_final.end(
            output=result.answer,
            usage={
                "input":  final.usage.prompt_tokens,
                "output": final.usage.completion_tokens
            }
        )

    except Exception as e:
        logger.error(f"final answer error: {e}")
        try:
            fallback = call_openai(
                client.beta.chat.completions.parse,
                model=FINAL_MODEL,
                messages=messages + [{
                    "role": "user",
                    "content": f"{df_context} Give final answer. Set plotly_code to empty string and chart_type to none."
                }],
                response_format=AgentAnswer
            )
            if fallback is None:
                return None
            result = fallback.choices[0].message.parsed
            logger.warning("fallback answer used — no chart")
        except Exception as e2:
            logger.error(f"fallback failed: {e2}")
            trace.update(output=f"final answer error: {e2}")
            return None

    # ── Memory ────────────────────────────────────────────────────
    memory.add_answer(result.answer)

    duration = round(time.time() - run_start, 2)
    logger.info(f"answer: {result.answer[:100]}...")
    logger.info(f"chart:  {result.chart_type}")
    logger.info(f"duration: {duration}s")

    # ── Trace update — no token fields, langfuse has them ─────────
    trace.update(
        output=result.answer,
        metadata={
            "chart_type":       result.chart_type,
            "duration_seconds": duration,
            "steps":            step,
            "sql_used":         result.sql_used
        }
    )
    langfuse.flush()

    # ── Evaluation ────────────────────────────────────────────────
    try:
        eval_result = evaluate_run(
            question   = question,
            answer     = result.answer,
            sql_used   = result.sql_used,
            chart_type = result.chart_type,
            steps      = step
        )

        trace.score(name="answer_relevance",      value=eval_result.answer_relevance)
        trace.score(name="sql_correctness",       value=eval_result.sql_correctness)
        trace.score(name="chart_appropriateness", value=eval_result.chart_appropriateness)
        trace.score(name="tool_efficiency",       value=eval_result.tool_efficiency)

        langfuse.flush()

        logger.info(f"eval — relevance: {eval_result.answer_relevance} | sql: {eval_result.sql_correctness} | chart: {eval_result.chart_appropriateness} | efficiency: {eval_result.tool_efficiency}")
        logger.debug(f"eval reasoning: {eval_result.reasoning}")

    except Exception as e:
        logger.warning(f"evaluation failed: {e}")

    # ── Chart ─────────────────────────────────────────────────────
    if result.plotly_code and result.chart_type != "none" and last_df is not None:
        try:
            local_vars = {"df": last_df, "px": px, "go": go}
            exec(result.plotly_code, local_vars)
            local_vars["fig"].show()
        except Exception as e:
            logger.warning(f"chart error: {e}")

    return result

In [ ]:
run_agent_with_cost("which few product has the most transactions?")


In [ ]:
conn.close()

In [3]:
import subprocess
import os
import signal

def kill_duckdb_processes(db_path: str):
    """Finds and kills any process currently locking the specified DuckDB file."""
    try:
        # 1. Run lsof -t to get only the PIDs of processes holding the file
        pids = subprocess.check_output(["lsof", "-t", db_path]).decode().split()

        for pid in pids:
            pid = int(pid)
            # 2. Don't kill the current process (safety check)
            if pid != os.getpid():
                print(f"[DuckDB] Killing process {pid} locking {db_path}")
                os.kill(pid, signal.SIGKILL) # Equivalent to kill -9

    except subprocess.CalledProcessError:
        # lsof returns a non-zero exit code if no files are found
        print(f"[DuckDB] No active locks found for {db_path}")
    except Exception as e:
        print(f"[DuckDB] Error clearing locks: {e}")

# Usage for your agent:
from config import DB_PATH
kill_duckdb_processes(DB_PATH)

[DuckDB] No active locks found for data/AdventureWorks.duckdb


lsof: status error on data/AdventureWorks.duckdb: No such file or directory
lsof 4.91
 latest revision: ftp://lsof.itap.purdue.edu/pub/tools/unix/lsof/
 latest FAQ: ftp://lsof.itap.purdue.edu/pub/tools/unix/lsof/FAQ
 latest man page: ftp://lsof.itap.purdue.edu/pub/tools/unix/lsof/lsof_man
 usage: [-?abhlnNoOPRtUvVX] [+|-c c] [+|-d s] [+D D] [+|-f[cgG]]
 [-F [f]] [-g [s]] [-i [i]] [+|-L [l]] [+|-M] [-o [o]] [-p s]
 [+|-r [t]] [-s [p:s]] [-S [t]] [-T [t]] [-u s] [+|-w] [-x [fl]] [--] [names]
Use the ``-h'' option to get more help information.
